In [121]:
from tqdm.auto import tqdm
import numpy as np
from dotenv import load_dotenv
import importlib
import os
from evaluation_utils import map_progress, calc_total_price, llm_structured_retry, RAGWithUsage, calc_price
import sys
from embedder import Embedder
import pandas as pd
from pydantic import BaseModel, Field
from ingest import build_index
from rag_helper import RAGBase, llm_client
import json
from gitsource import chunk_documents
from minsearch import VectorSearch, Index

load_dotenv()

True

In [115]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]

In [85]:
documents[0]

{'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phone uses a simp

In [86]:
data_gen_instructions = """
You emulate a student who is taking our LLM course.
You are given one lesson page from the course.
Formulate 5 questions this student might ask that are answered by this page.

Rules:
- The page should contain the answer to each question.
- Make the questions complete and not too short.
- Use as few words as possible from the page; don't copy its phrasing.
- The questions should resemble how people actually ask things online:
  not too formal, not too short, not too long.
- Ask about the content of the lesson, not about its formatting or filename.
""".strip()

In [105]:
class Questions(BaseModel):
    questions: list[str]

In [103]:
# Generate questions
def generate_ground_truth(doc):
    user_prompt = json.dumps(doc)

    # Use the llm with the retry
    out, usage = llm_structured_retry(
        llm_client,
        data_gen_instructions,
        user_prompt,
        Questions
    )

    # Create set for the questions for each answer
    results = []

    for q in out.questions:
        results.append({
            "question": q,
            "content": doc["content"],
            "filename": doc["filename"]
        })

    return results, usage

    results

## Q1. Generating questions

In [106]:
# Create set for the questions and answers
# generate_ground_truth returns two things for each document: the generated records and the token usage.
# Split those into separate lists:
ground_truth = []
usages = []

for doc in tqdm(documents[:3]):
    records, usage = generate_ground_truth(doc)
    ground_truth.extend(records)
    usages.append(usage)

  0%|          | 0/3 [00:00<?, ?it/s]

In [107]:
# Calculate the cost
cost = calc_price(usage)

cost

{'input_cost': 0.00131475,
 'output_cost': 0.0004455,
 'total_cost': 0.0017602499999999999}

In [108]:
usage.input_tokens

1753

In [111]:
# Upload ready made ground truth
df_ground_truth = pd.read_csv("ground-truth-data.csv")
ground_truth = df_ground_truth.to_dict(orient="records")
len(ground_truth)

395

In [116]:
# chunk documents
chunks = chunk_documents(documents, size=2000, step=1000)
len(chunks)

295

In [117]:
embed = Embedder()

In [118]:
# Embed in batches
batch_size = 50
X = []

for i in tqdm(range(0, len(chunks), batch_size)):
    batch = [chunk['content'] for chunk in chunks[i:i + batch_size]]
    batch_vectors = embed.encode_batch(batch)
    X.extend(batch_vectors)

X = np.array(X)

  0%|          | 0/6 [00:00<?, ?it/s]

In [122]:
# Create text index
Index = Index(
    text_fields=['content'],
    keyword_fields=['filename']
)

Index.fit(chunks)

In [136]:
# define text search function
def text_search(query, num):
    
    return Index.search(
        query,
        num_results=num
    )

In [123]:
# Create vector index
vindex = VectorSearch(keyword_fields=['filename'])
# Similar to dot (also scalar multiplication)
vindex.fit(X, chunks)

In [139]:
# define vector search function
def vector_search(query, num):

    query_vector = embed.encode(query)
    
    return vindex.search(
    query_vector,
    num_results=num
)


In [127]:
# hybrid search function
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

In [128]:
# define hybrid search function
def hybrid_search(query, k=60):
    text_results = text_search(query, num_results=10)
    vector_results = vector_search(query, num_results=10)
    return rrf([text_results, vector_results], k=k)

## Q2. First result with text search

In [129]:
q = ground_truth[0]["question"]
q

'Can I take this course at my own pace and still receive a certificate at the end?'

In [137]:
result = text_search(q, 1)
result

[{'start': 2000,
  'content': 'you want to receive a certificate, you need to submit your project while we\'re still accepting submissions.\n\nCourse: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?\nYou don\'t need it. You\'re accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.\n\nWhat is the video/zoom link to the stream for the "Office Hours" or live/workshop sessions?\nThe zoom link is only published to instructors/presenters/TAs. Students participate via YouTube Live and submit questions to Slido.\n\nCloud alternatives with GPU\nCheck the quota and reset cycle carefully. Potential options include Google Colab, Kaggle, Databricks.\n"""\n```\n\nNotice the prompt doesn\'t end with `Answer:`. With older models like\nGPT-3 we added that to nudge the model into completing the

## Q3. First result with vector search

In [140]:
resultV = vector_search(q, 1)
resultV

[{'start': 2000,
  'content': 'you want to receive a certificate, you need to submit your project while we\'re still accepting submissions.\n\nCourse: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?\nYou don\'t need it. You\'re accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.\n\nWhat is the video/zoom link to the stream for the "Office Hours" or live/workshop sessions?\nThe zoom link is only published to instructors/presenters/TAs. Students participate via YouTube Live and submit questions to Slido.\n\nCloud alternatives with GPU\nCheck the quota and reset cycle carefully. Potential options include Google Colab, Kaggle, Databricks.\n"""\n```\n\nNotice the prompt doesn\'t end with `Answer:`. With older models like\nGPT-3 we added that to nudge the model into completing the

## Q4. Evaluating text search

In [153]:
def compute_relevance(q, search_function, num):
    doc_id = q["document"]
    results = search_function(q["question"], num)

    relevance = []
    for d in results:
        relevance.append(int(d["filename"] == doc_id))

    return relevance

In [154]:
# total relevance function gets a search_function too
def compute_relevance_total(ground_truth, search_function, num):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance(q, search_function, num)
        relevance_total.append(relevance)

    return relevance_total

In [155]:
relevance_total = compute_relevance_total(ground_truth, text_search, 5)
relevance_total

  0%|          | 0/395 [00:00<?, ?it/s]

[[0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0,

Each line is one query. If a line contains 1, search found the correct document somewhere in the top 5 results. If the line contains only zeros, search did not find the correct document.

In [158]:
# For all documents
num = 5
relevance_total = compute_relevance_total(ground_truth, text_search, num)

  0%|          | 0/395 [00:00<?, ?it/s]

## Hit rate
Did the search provide the correct answer anywhere in the top 5 results?

In [159]:
# Calculatethe hit rate for this example
cnt = 0

for line in relevance_total:
    if 1 in line:
        cnt = cnt + 1

len(relevance_total), cnt

(395, 0)

In [160]:
print("Hit rate = " + str(100*cnt/len(relevance_total)) + '%')

Hit rate = 0.0%


In [161]:
# Hit rate function
def hit_rate(relevance):
    cnt = 0

    for line in relevance:
        if 1 in line:
            cnt = cnt + 1

    return cnt / len(relevance)

In [162]:
hit_rate(relevance_total)

0.0